In [27]:
import pandas as pd
import ollama
import re

In [9]:
prova_2015 = pd.read_csv("prova_2015.csv")
anos = prova_2015['Ano'].unique()
print(anos)
prova_2015.head()

[2015]


,Nquestao,enunciado,A,B,C,D,E,questao_completa,Ano,Aplicacao,gabarito
0,1,"Suponha que, entabulado contrato facultativo d...",O prazo prescricional anual é interrompido com...,O prazo prescricional anual é interrompido com...,O prazo prescricional trienal é suspenso com o...,O prazo prescricional anual é suspenso com o p...,"Na hipótese de resseguro, o prazo prescriciona...","Suponha que, entabulado contrato facultativo d...",2015,2,D
1,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,A
2,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,C
3,3,A respeito da hermenêutica e da aplicação do d...,Diante da existência de antinomia entre dois d...,"Os tradicionais critérios hierárquico, cronoló...",A técnica da subsunção é suficiente e adequada...,Diante da existência de antinomia entre dois d...,"A aplicação do princípio da especialidade, em ...",A respeito da hermenêutica e da aplicação do d...,2015,2,A
4,4,A respeito da locação dos imóveis urbanos da L...,"Conforme entendimento consolidado do STJ, o pr...",É assente na jurisprudência do STJ que a cláus...,"Nas locações comerciais, exige-se a anuência d...","Conforme entendimento do STF, a penhora de bem...","A responsabilidade dos fiadores, no caso de pr...",A respeito da locação dos imóveis urbanos da L...,2015,2,C


In [14]:
SYSTEM = (
    """Você é um especialista em Direito brasileiro resolvendo questões de 
    múltipla escolha de provas. Considere a legislação e a 
    jurisprudência brasileiras. Apenas uma alternativa está correta."""
)

TEMPLATE = """Responda a questão abaixo.

Enunciado: {enunciado}
Alternativas: {alternativas}

Responda apenas com a letra da alternativa correta, exatamente neste formato:
Resposta: <letra>"""



In [22]:
def montar_prompt(row, letras="ABCDE"):
    alts = "\n".join(f"{l}) {row[l]}" for l in letras if pd.notna(row.get(l)))
    return TEMPLATE.format(enunciado=row["enunciado"], alternativas=alts)

In [28]:
def extrair_letra(texto):
    achados = re.findall(r"Resposta:\s*\**\s*([A-E])", texto, flags=re.I)
    return achados[-1].upper() if achados else None

In [39]:
def usou_gpu(modelo):
    for m in ollama.ps().models:
        if m.model == modelo:
            return m.size_vram > 0
    return None

In [30]:
x = extrair_letra("Resposta: ** A")
x

'A'

In [ ]:
def perguntar(modelo, prompt):
    r = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0, "seed": 42, "num_ctx": 4096},
        keep_alive="30m",
    )
    texto = r["message"]["content"]
    pct_gpu, tamanho_gb = usou_gpu(modelo)  
    return {
        "resposta_bruta": texto,
        "letra": extrair_letra(texto),
        "tempo_s": r["total_duration"] / 1e9,
        "tokens_prompt": r["prompt_eval_count"],
        "tokens_saida": r["eval_count"],
        "usou_gpu": usou_gpu(modelo),
    }

In [42]:
# escolha o modelo
modelo = "gemma4:e4b-it-q4_K_M"
resultados = []

for _, row in prova_2015.iterrows():
        out = perguntar(modelo, montar_prompt(row))
        resultados.append({"modelo": modelo, "Nquestao": row["Nquestao"], **out})

  
    
df_res = pd.DataFrame(resultados)

In [45]:
df_final = prova_2015.merge(df_res, on="Nquestao", how="left")
df_final["absoluto_correto"] = df_final["letra"] == df_final["gabarito"]
df_final["absoluto_correto"].sum() / len(df_final) * 100

np.float64(30.097087378640776)

In [50]:
df_final.head(50)

,Nquestao,enunciado,A,B,C,D,E,questao_completa,Ano,Aplicacao,gabarito,modelo,resposta_bruta,letra,tempo_s,tokens_prompt,tokens_saida,usou_gpu,absoluto_correto
0,1,"Suponha que, entabulado contrato facultativo d...",O prazo prescricional anual é interrompido com...,O prazo prescricional anual é interrompido com...,O prazo prescricional trienal é suspenso com o...,O prazo prescricional anual é suspenso com o p...,"Na hipótese de resseguro, o prazo prescriciona...","Suponha que, entabulado contrato facultativo d...",2015,2,D,gemma4:e4b-it-q4_K_M,Resposta: A,A,8.245840,376,1126,True,False
1,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,A,gemma4:e4b-it-q4_K_M,Resposta: A,A,5.812732,285,828,True,True
2,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,A,gemma4:e4b-it-q4_K_M,Resposta: A,A,5.832850,285,803,True,True
3,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,C,gemma4:e4b-it-q4_K_M,Resposta: A,A,5.812732,285,828,True,False
4,2,No que se refere ao contrato estimatório do Di...,"Pode ter por objeto bem fungível, e a restitui...","Os riscos são do consignante, que suporta a pe...","Após a entrega da coisa, a posse é exercida em...",O preço de estima é ato unilateral do consigna...,Em decorrência da natureza própria do contrato...,No que se refere ao contrato estimatório do Di...,2015,2,C,gemma4:e4b-it-q4_K_M,Resposta: A,A,5.832850,285,803,True,False
5,3,A respeito da hermenêutica e da aplicação do d...,Diante da existência de antinomia entre dois d...,"Os tradicionais critérios hierárquico, cronoló...",A técnica da subsunção é suficiente e adequada...,Diante da existência de antinomia entre dois d...,"A aplicação do princípio da especialidade, em ...",A respeito da hermenêutica e da aplicação do d...,2015,2,A,gemma4:e4b-it-q4_K_M,Resposta: A,A,7.168110,336,967,True,True
6,4,A respeito da locação dos imóveis urbanos da L...,"Conforme entendimento consolidado do STJ, o pr...",É assente na jurisprudência do STJ que a cláus...,"Nas locações comerciais, exige-se a anuência d...","Conforme entendimento do STF, a penhora de bem...","A responsabilidade dos fiadores, no caso de pr...",A respeito da locação dos imóveis urbanos da L...,2015,2,C,gemma4:e4b-it-q4_K_M,Resposta: C,C,8.473936,386,1177,True,True
7,5,"No que se refere às pessoas, assinale a opção ...",A declaração de ausência é a condição eficient...,"Está consolidado o entendimento, na doutrina e...",Dentre as pessoas jurídicas de direito público...,"Conforme entendimento prevalente do STJ, a dis...",A emancipação voluntária dos pais é ato revogá...,"No que se refere às pessoas, assinale a opção ...",2015,2,D,gemma4:e4b-it-q4_K_M,Resposta: D,D,8.584856,317,1043,True,True
8,6,"Em atenção ao direito das obrigações, assinale...","Se há assunção cumulativa, compreende-se como ...",A multa moratória e a multa compensatória pode...,A obrigação portável (portable) é aquela em qu...,"Na solidariedade passiva, a renúncia e a remis...","Na assunção de dívida, a oposição da exceção d...","Em atenção ao direito das obrigações, assinale...",2015,2,E,gemma4:e4b-i

In [51]:
df_final.to_csv("resultados_prova_2015.csv", index=False)